# Step 6: Validate Neo4j Knowledge Graph

This notebook validates the knowledge graph loaded into Neo4j:
1. Check node counts: Food, Ingredient, Brand, Disease
2. Check relationship counts: CONTAINS_INGREDIENT, HAS_BRAND, RELATES_TO
3. Sample paths: food → ingredient → disease

In [1]:
# Import required libraries
from neo4j import GraphDatabase
import pandas as pd

In [3]:
# Neo4j Configuration
NEO4J_URI = "neo4j://127.0.0.1:7687"  
NEO4J_USER = "neo4j"                 
NEO4J_PASSWORD = "password"  
# Connect to Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Helper function to run queries
def run_query(query):
    with driver.session() as session:
        result = session.run(query)
        return [dict(record) for record in result]

# Test connection
try:
    run_query("RETURN 1 AS connected")
    print("✅ Connected to Neo4j successfully!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to Neo4j successfully!


## 1. Node Counts

In [4]:
print("="*50)
print("NODE COUNTS")
print("="*50)

nodes = run_query("MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC")

total_nodes = 0
for n in nodes:
    label = n['label'][0] if n['label'] else 'Unknown'
    count = n['count']
    total_nodes += count
    print(f"  {label:20} : {count:>12,}")

print(f"  {'-'*35}")
print(f"  {'TOTAL':20} : {total_nodes:>12,}")

NODE COUNTS
  Food                 :    1,977,398
  Ingredient           :      225,440
  Brand                :       37,033
  Disease              :        2,324
  -----------------------------------
  TOTAL                :    2,242,195


In [5]:
# Specific node counts
print("\nDetailed Node Counts:")

food_count = run_query("MATCH (f:Food) RETURN count(f) AS count")[0]['count']
ingredient_count = run_query("MATCH (i:Ingredient) RETURN count(i) AS count")[0]['count']
brand_count = run_query("MATCH (b:Brand) RETURN count(b) AS count")[0]['count']
disease_count = run_query("MATCH (d:Disease) RETURN count(d) AS count")[0]['count']

print(f"  Food nodes:       {food_count:>12,}")
print(f"  Ingredient nodes: {ingredient_count:>12,}")
print(f"  Brand nodes:      {brand_count:>12,}")
print(f"  Disease nodes:    {disease_count:>12,}")


Detailed Node Counts:
  Food nodes:          1,977,398
  Ingredient nodes:      225,440
  Brand nodes:            37,033
  Disease nodes:           2,324


## 2. Relationship Counts

In [6]:
print("="*50)
print("RELATIONSHIP COUNTS")
print("="*50)

rels = run_query("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC")

total_rels = 0
for r in rels:
    rel_type = r['type']
    count = r['count']
    total_rels += count
    print(f"  {rel_type:25} : {count:>12,}")

print(f"  {'-'*40}")
print(f"  {'TOTAL':25} : {total_rels:>12,}")

RELATIONSHIP COUNTS
  CONTAINS_INGREDIENT       :   16,353,259
  HAS_BRAND                 :    1,977,398
  RELATES_TO                :       14,242
  ----------------------------------------
  TOTAL                     :   18,344,899


## 3. Graph Statistics

In [7]:
print("="*50)
print("GRAPH STATISTICS")
print("="*50)

# Average relationships per Food
avg_rels = run_query("""
    MATCH (f:Food) 
    OPTIONAL MATCH (f)-[r]-() 
    WITH f, count(r) AS degree 
    RETURN avg(degree) AS avg_degree, min(degree) AS min_degree, max(degree) AS max_degree
""")
if avg_rels:
    print(f"\nFood Node Degree:")
    print(f"  Average: {avg_rels[0]['avg_degree']:.2f}")
    print(f"  Min:     {avg_rels[0]['min_degree']}")
    print(f"  Max:     {avg_rels[0]['max_degree']}")

# Average relationships per Ingredient
ing_stats = run_query("""
    MATCH (i:Ingredient)
    OPTIONAL MATCH (i)<-[r:CONTAINS_INGREDIENT]-()
    WITH i, count(r) AS degree
    RETURN avg(degree) AS avg_degree, max(degree) AS max_degree
""")
if ing_stats:
    print(f"\nIngredient Node Degree (CONTAINS_INGREDIENT):")
    print(f"  Average: {ing_stats[0]['avg_degree']:.2f}")
    print(f"  Max:     {ing_stats[0]['max_degree']}")

GRAPH STATISTICS

Food Node Degree:
  Average: 9.27
  Min:     1
  Max:     112

Ingredient Node Degree (CONTAINS_INGREDIENT):
  Average: 72.54
  Max:     606942


In [8]:
# Unique entities
print("\nUnique Entities:")

unique_ings = run_query("MATCH (i:Ingredient) RETURN count(DISTINCT i.name) AS count")[0]['count']
unique_brands = run_query("MATCH (b:Brand) RETURN count(DISTINCT b.name) AS count")[0]['count']
unique_diseases = run_query("MATCH (d:Disease) RETURN count(DISTINCT d.name) AS count")[0]['count']

print(f"  Unique Ingredients: {unique_ings:,}")
print(f"  Unique Brands:      {unique_brands:,}")
print(f"  Unique Diseases:    {unique_diseases:,}")


Unique Entities:
  Unique Ingredients: 225,440
  Unique Brands:      37,033
  Unique Diseases:    2,324


## 4. Sample Paths

In [9]:
print("="*50)
print("SAMPLE PATHS: Food → Ingredient")
print("="*50)

food_ing_samples = run_query("""
    MATCH (f:Food)-[:CONTAINS_INGREDIENT]->(i:Ingredient)
    RETURN f.name AS food, i.name AS ingredient
    LIMIT 10
""")

for s in food_ing_samples:
    food = s['food'][:50] + '...' if len(s['food']) > 50 else s['food']
    print(f"  {food} → {s['ingredient']}")

SAMPLE PATHS: Food → Ingredient
  CHEDDAR CHEESE BITE SIZE SANDWICH CRACKER, CHEDDAR... → VEGETABLE OIL
  Cape Cod Potato Chips, Lightly Salted Original Ket... → VEGETABLE OIL
  Cape Cod Potato Chips, Sweet Mesquite Barbeque Ket... → VEGETABLE OIL
  Cape Cod Sweet & Spicy Jalapeno Chips, 2 Oz Bag → VEGETABLE OIL
  Cape Cod Waves Potato Chips, Wavy Cut Sea Salt Ket... → VEGETABLE OIL
  Cape Cod Potato Chips, Original Kettle Chips, 1.5 ... → VEGETABLE OIL
  Cape Cod Sea Salt & Vinegar Chips, 7.5 Oz Bag → VEGETABLE OIL
  Cape Cod Waves Potato Chips, Wavy Cut Jalapeno Ran... → VEGETABLE OIL
  Cape Cod Less Fat White Cheddar & Sour Cream Chips... → VEGETABLE OIL
  Cape Cod Potato Chips, Sea Salt & Vinegar Kettle C... → VEGETABLE OIL


In [10]:
print("\n" + "="*50)
print("SAMPLE PATHS: Ingredient → Disease")
print("="*50)

ing_disease_samples = run_query("""
    MATCH (i:Ingredient)-[r:RELATES_TO]->(d:Disease)
    RETURN i.name AS ingredient, d.name AS disease, r.pmid AS pmid
    LIMIT 10
""")

if ing_disease_samples:
    for s in ing_disease_samples:
        print(f"  {s['ingredient']} → {s['disease']} (PMID: {s['pmid']})")
else:
    print("  No Ingredient → Disease relationships found")


SAMPLE PATHS: Ingredient → Disease
  dark chocolate → diabetes, insulin resistance, cardiovascular disease (PMID: 24100673)
  protein → diabetes, insulin resistance, cardiovascular disease (PMID: 41390803)
  glucose → diabetes, insulin resistance, cardiovascular disease (PMID: 41390803)
  erythritol → alzheimer (PMID: 34069842)
  resveratrol → alzheimer (PMID: 41375150)
  black pepper → alzheimer (PMID: 34235220)
  probiotic → alzheimer (PMID: 41375975)
  caffeine → alzheimer (PMID: 40945382)
  pork → alzheimer (PMID: 37781803)
  beer → alzheimer (PMID: 40713707)


In [11]:
print("\n" + "="*50)
print("SAMPLE PATHS: Food → Ingredient → Disease")
print("="*50)

full_path_samples = run_query("""
    MATCH (f:Food)-[:CONTAINS_INGREDIENT]->(i:Ingredient)-[:RELATES_TO]->(d:Disease)
    RETURN f.name AS food, i.name AS ingredient, d.name AS disease
    LIMIT 10
""")

if full_path_samples:
    for s in full_path_samples:
        food = s['food'][:30] + '...' if len(s['food']) > 30 else s['food']
        print(f"  {food} → {s['ingredient']} → {s['disease']}")
else:
    print("  No full paths found (Food → Ingredient → Disease)")
    print("  This may happen if ingredient names don't match between USDA and PubMed data")


SAMPLE PATHS: Food → Ingredient → Disease
  No full paths found (Food → Ingredient → Disease)
  This may happen if ingredient names don't match between USDA and PubMed data


In [12]:
print("\n" + "="*50)
print("SAMPLE PATHS: Food → Brand")
print("="*50)

food_brand_samples = run_query("""
    MATCH (f:Food)-[:HAS_BRAND]->(b:Brand)
    RETURN f.name AS food, b.name AS brand
    LIMIT 10
""")

for s in food_brand_samples:
    food = s['food'][:40] + '...' if len(s['food']) > 40 else s['food']
    print(f"  {food} → {s['brand']}")


SAMPLE PATHS: Food → Brand
  WESSON Corn Oil 1 GAL → Richardson Oilseed Products (US) Limited
  BEYOND BUTTER previously Move Over Butte... → Richardson Oilseed Products (US) Limited
  WESSON Vegetable Oil 5 QT → Richardson Oilseed Products (US) Limited
  WESSON Vegetable Oil 16 FL OZ → Richardson Oilseed Products (US) Limited
  WESSON Canola Oil 5 QT → Richardson Oilseed Products (US) Limited
  WESSON Vegetable Oil 24 FL OZ → Richardson Oilseed Products (US) Limited
  BEYOND BUTTER previously Move Over Butte... → Richardson Oilseed Products (US) Limited
  WESSON Vegetable Oil 64 FL OZ → Richardson Oilseed Products (US) Limited
  WESSON Vegetable Oil 1 GAL → Richardson Oilseed Products (US) Limited
  WESSON Best Blend Oil 48 FL OZ → Richardson Oilseed Products (US) Limited


## 5. Top Entities

In [13]:
print("="*50)
print("TOP 10 MOST COMMON INGREDIENTS")
print("="*50)

top_ingredients = run_query("""
    MATCH (i:Ingredient)<-[:CONTAINS_INGREDIENT]-(f:Food)
    RETURN i.name AS ingredient, count(f) AS food_count
    ORDER BY food_count DESC
    LIMIT 10
""")

for idx, ing in enumerate(top_ingredients, 1):
    print(f"  {idx:2}. {ing['ingredient']:30} ({ing['food_count']:,} foods)")

TOP 10 MOST COMMON INGREDIENTS
   1. WATER                          (606,942 foods)
   2. SALT                           (594,546 foods)
   3. SUGAR                          (593,679 foods)
   4. CITRIC ACID                    (236,311 foods)
   5. CORN SYRUP                     (195,497 foods)
   6. NATURAL FLAVOR                 (139,899 foods)
   7. SEA SALT                       (135,928 foods)
   8. NATURAL FLAVORS                (119,791 foods)
   9. SOYBEAN OIL                    (108,535 foods)
  10. SPICES                         (107,574 foods)


In [14]:
print("\n" + "="*50)
print("TOP 10 BRANDS BY FOOD COUNT")
print("="*50)

top_brands = run_query("""
    MATCH (b:Brand)<-[:HAS_BRAND]-(f:Food)
    RETURN b.name AS brand, count(f) AS food_count
    ORDER BY food_count DESC
    LIMIT 10
""")

for idx, brand in enumerate(top_brands, 1):
    print(f"  {idx:2}. {brand['brand'][:40]:40} ({brand['food_count']:,} foods)")


TOP 10 BRANDS BY FOOD COUNT
   1. Wal-Mart Stores, Inc.                    (46,915 foods)
   2. Target Stores                            (44,306 foods)
   3. Meijer, Inc.                             (31,231 foods)
   4. Safeway, Inc.                            (29,484 foods)
   5. GENERAL MILLS SALES INC.                 (28,399 foods)
   6. Topco Associates, Inc.                   (26,163 foods)
   7. The Kroger Co.                           (23,636 foods)
   8. Hy-Vee, Inc.                             (23,176 foods)
   9. Supervalu, Inc.                          (18,256 foods)
  10. Unknown                                  (17,232 foods)


In [15]:
print("\n" + "="*50)
print("TOP 10 DISEASES BY INGREDIENT CONNECTIONS")
print("="*50)

top_diseases = run_query("""
    MATCH (d:Disease)<-[:RELATES_TO]-(i:Ingredient)
    RETURN d.name AS disease, count(i) AS ingredient_count
    ORDER BY ingredient_count DESC
    LIMIT 10
""")

if top_diseases:
    for idx, disease in enumerate(top_diseases, 1):
        print(f"  {idx:2}. {disease['disease']:30} ({disease['ingredient_count']:,} ingredients)")
else:
    print("  No disease data found")


TOP 10 DISEASES BY INGREDIENT CONNECTIONS
   1. cancer                         (212 ingredients)
   2. inflammation                   (210 ingredients)
   3. diabetes                       (189 ingredients)
   4. obesity                        (187 ingredients)
   5. cancer, breast cancer          (181 ingredients)
   6. depression                     (169 ingredients)
   7. colorectal cancer, cancer      (158 ingredients)
   8. cardiovascular disease         (150 ingredients)
   9. diabetes, diabetes mellitus    (149 ingredients)
  10. diabetes, type 2 diabetes      (149 ingredients)


## 6. Validation Summary

In [16]:
print("\n" + "="*60)
print("KNOWLEDGE GRAPH VALIDATION SUMMARY")
print("="*60)

# Validation checks
checks = []

# Check 1: Food nodes exist
if food_count > 0:
    checks.append(("✅", f"Food nodes: {food_count:,}"))
else:
    checks.append(("❌", "No Food nodes found"))

# Check 2: Ingredient nodes exist
if ingredient_count > 0:
    checks.append(("✅", f"Ingredient nodes: {ingredient_count:,}"))
else:
    checks.append(("❌", "No Ingredient nodes found"))

# Check 3: Brand nodes exist
if brand_count > 0:
    checks.append(("✅", f"Brand nodes: {brand_count:,}"))
else:
    checks.append(("⚠️", "No Brand nodes found"))

# Check 4: Disease nodes exist
if disease_count > 0:
    checks.append(("✅", f"Disease nodes: {disease_count:,}"))
else:
    checks.append(("⚠️", "No Disease nodes found"))

# Check 5: CONTAINS_INGREDIENT relationships
contains_count = run_query("MATCH ()-[r:CONTAINS_INGREDIENT]->() RETURN count(r) AS count")[0]['count']
if contains_count > 0:
    checks.append(("✅", f"CONTAINS_INGREDIENT relationships: {contains_count:,}"))
else:
    checks.append(("❌", "No CONTAINS_INGREDIENT relationships"))

# Check 6: HAS_BRAND relationships
brand_rel_count = run_query("MATCH ()-[r:HAS_BRAND]->() RETURN count(r) AS count")[0]['count']
if brand_rel_count > 0:
    checks.append(("✅", f"HAS_BRAND relationships: {brand_rel_count:,}"))
else:
    checks.append(("⚠️", "No HAS_BRAND relationships"))

# Check 7: RELATES_TO relationships
relates_count = run_query("MATCH ()-[r:RELATES_TO]->() RETURN count(r) AS count")[0]['count']
if relates_count > 0:
    checks.append(("✅", f"RELATES_TO relationships: {relates_count:,}"))
else:
    checks.append(("⚠️", "No RELATES_TO relationships"))

# Print results
for status, message in checks:
    print(f"  {status} {message}")

# Overall status
failed = sum(1 for s, _ in checks if s == "❌")
warnings = sum(1 for s, _ in checks if s == "⚠️")
passed = sum(1 for s, _ in checks if s == "✅")

print(f"\n  Summary: {passed} passed, {warnings} warnings, {failed} failed")

if failed == 0:
    print("\n  🎉 Knowledge Graph validation PASSED!")
else:
    print("\n  ⚠️ Knowledge Graph has issues - please check above")


KNOWLEDGE GRAPH VALIDATION SUMMARY
  ✅ Food nodes: 1,977,398
  ✅ Ingredient nodes: 225,440
  ✅ Brand nodes: 37,033
  ✅ Disease nodes: 2,324
  ✅ CONTAINS_INGREDIENT relationships: 16,353,259
  ✅ HAS_BRAND relationships: 1,977,398
  ✅ RELATES_TO relationships: 14,242

  Summary: 7 passed, 0 warnings, 0 failed

  🎉 Knowledge Graph validation PASSED!


In [17]:
# Close connection
driver.close()
print("\n✅ Neo4j connection closed")
print("\nStep 6: Validation Complete!")


✅ Neo4j connection closed

Step 6: Validation Complete!
